# 0927 & 0928 CT 비교 (Vision1~2)

이 노트북은 아래 사양(첨부 PDF)을 그대로 구현하여,
- Vision1~2 생산량(주간/야간)
- CT(연속 생산 간 시간차 기반, 600초 이상 제외 후 boxplot 통계)
를 날짜/주야간/설비별로 산출합니다.


In [1]:
# 필수 패키지
import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [2]:
# ============================================
# 0) 사용자 설정
# ============================================

# DB 접속 정보
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

# 대상 "교대일자(D-DAY)" (야간은 D 20:30~23:59 + D+1 00:00~08:29)
# 예: 2025-09-27 주간/야간, 2025-09-28 주간/야간을 보고 싶으면 아래처럼 설정
TARGET_SHIFT_DATES = ["20250927", "20250928"]

# 제외할 교대일자(사양: 20250926 야간 생산 제외)
EXCLUDE_SHIFT_DATES = ["20250926"]

# 테이블
SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
TABLE  = "fct_vision_testlog_txt_processing_history"

# 필터(사양 고정)
STATIONS = ["Vision1", "Vision2"]
RESULT_FILTER = "PASS"
GOODORBAD_FILTER = "GoodFile"

# CT 차이(초) 상한 (사양: 600초 이상 제외)
CT_MAX_SEC_EXCLUSIVE = 600


In [3]:
# ============================================
# 1) 유틸: 날짜/시간 파싱 + 주야간/교대일자 계산
#    - 요구사항: 주야간 구분은 end_time 값(시간)으로만 판단
# ============================================

def parse_end_ts(end_day: pd.Series, end_time: pd.Series) -> pd.Series:
    """end_day(YYYYMMDD) + end_time(HH:MM:SS[.fff]) -> Timestamp"""
    s = end_day.astype(str).str.strip() + " " + end_time.astype(str).str.strip()
    ts = pd.to_datetime(s, format="%Y%m%d %H:%M:%S", errors="coerce")
    # ms/µs 포함 케이스 대비
    if ts.isna().any():
        ts2 = pd.to_datetime(s, format="%Y%m%d %H:%M:%S.%f", errors="coerce")
        ts = ts.fillna(ts2)
    return ts


def _parse_time_only(end_time: pd.Series) -> pd.Series:
    """end_time(HH:MM:SS[.fff]) -> datetime.time (Series)"""
    s = end_time.astype(str).str.strip()
    t = pd.to_datetime(s, format="%H:%M:%S", errors="coerce").dt.time
    if t.isna().any():
        t2 = pd.to_datetime(s, format="%H:%M:%S.%f", errors="coerce").dt.time
        # pandas time series fillna via object conversion
        t = t.astype(object)
        t[pd.isna(t)] = t2[pd.isna(t2) == False]
        t = t.astype(object)
    return t


def calc_day_night_and_shift_date(end_day: pd.Series, end_time: pd.Series) -> pd.DataFrame:
    """사양 기준으로 day/night 및 shift_date(D-DAY) 계산

    - day: 08:30:00 ~ 20:29:59
    - night: 20:30:00 ~ 08:29:59
    - shift_date: night 중 00:00~08:29는 전날(D-1)에 귀속
    - 요구사항: 주야간 판단은 end_time만 사용
    """
    t = _parse_time_only(end_time)
    day_start = pd.to_datetime("08:30:00").time()
    night_start = pd.to_datetime("20:30:00").time()

    is_day = (t >= day_start) & (t < night_start)
    day_night = np.where(is_day, "day", "night")

    # shift_date는 end_day를 기준으로, night & (end_time < 08:30) 일 때만 -1
    shift_date = pd.to_datetime(end_day.astype(str).str.strip(), format="%Y%m%d", errors="coerce")
    is_night_early = (day_night == "night") & (t < day_start)
    shift_date = np.where(is_night_early, shift_date - pd.Timedelta(days=1), shift_date)
    shift_date = pd.to_datetime(shift_date).strftime("%Y%m%d")

    return pd.DataFrame({"day_night_calc": day_night, "shift_date": shift_date})


def build_query_end_days(target_shift_dates: list[str]) -> list[str]:
    """야간(익일 08:29까지) 커버를 위해 max(target)+1일까지 조회"""
    dmin = pd.to_datetime(min(target_shift_dates), format="%Y%m%d")
    dmax = pd.to_datetime(max(target_shift_dates), format="%Y%m%d") + pd.Timedelta(days=1)
    days = pd.date_range(dmin, dmax, freq="D").strftime("%Y%m%d").tolist()
    return days


In [4]:
# ============================================
# 2) DB 조회 (스키마 자동 감지: day_night 컬럼이 없어도 동작)
# ============================================

QUERY_END_DAYS = build_query_end_days(TARGET_SHIFT_DATES)
print("QUERY_END_DAYS =", QUERY_END_DAYS)

engine = create_engine(
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

# 2-1) 테이블 컬럼 목록 조회
col_sql = text("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = :schema
  AND table_name   = :table
ORDER BY ordinal_position
""")

with engine.begin() as conn:
    cols = pd.read_sql(col_sql, conn, params={"schema": SCHEMA, "table": TABLE})["column_name"].tolist()

print("Detected columns:", cols)

# 2-2) 필요한 컬럼 구성 (day_night는 있으면 가져오고, 없으면 계산값으로만 처리)
base_cols = ["id", "station", "end_day", "end_time", "result", "goodorbad"]
select_cols = [c for c in base_cols if c in cols]
if "day_night" in cols:
    select_cols.insert(4, "day_night")  # end_time 다음 위치(가독성)

if set(["id","station","end_day","end_time"]).issubset(set(select_cols)) is False:
    raise RuntimeError(f"필수 컬럼(id,station,end_day,end_time) 중 누락이 있습니다. 현재 선택: {select_cols}")

select_sql = ",\n    ".join([f'"{c}"' for c in select_cols])

sql = text(f"""
SELECT
    {select_sql}
FROM "{SCHEMA}"."{TABLE}"
WHERE "end_day" = ANY(:end_days)
  AND "station" = ANY(:stations)
  AND "result" = :result_filter
  AND "goodorbad" = :goodorbad_filter
ORDER BY "end_day" ASC, "end_time" ASC
""")

with engine.begin() as conn:
    df = pd.read_sql(sql, conn, params={
        "end_days": QUERY_END_DAYS,
        "stations": STATIONS,
        "result_filter": RESULT_FILTER,
        "goodorbad_filter": GOODORBAD_FILTER,
    })

print("rows fetched =", len(df))
df.head()


QUERY_END_DAYS = ['20250927', '20250928', '20250929']
Detected columns: ['id', 'barcode_information', 'station', 'end_day', 'end_time', 'remark', 'result', 'goodorbad', 'filename', 'file_path', 'processed_at']
rows fetched = 12625


,id,station,end_day,end_time,result,goodorbad
0,5953706,Vision2,20250927,00:00:10,PASS,GoodFile
1,5954014,Vision1,20250927,00:00:20,PASS,GoodFile
2,5953705,Vision2,20250927,00:00:29,PASS,GoodFile
3,5954013,Vision1,20250927,00:00:35,PASS,GoodFile
4,5953704,Vision2,20250927,00:00:45,PASS,GoodFile


In [5]:
# ============================================
# 3) 주야간/교대일자 계산 + 필터(사양: 20250926 야간 제외)
# ============================================

df["end_ts"] = parse_end_ts(df["end_day"], df["end_time"])

aux = calc_day_night_and_shift_date(df["end_day"], df["end_time"])
df = pd.concat([df, aux], axis=1)

# 참고: 원본 컬럼 day_night와 계산값이 다르면 계산값을 우선으로 사용
df["day_night_src"] = df.get("day_night", np.nan)
mismatch = (df["day_night_src"].notna()) & (df["day_night_src"].astype(str).str.lower() != df["day_night_calc"])
if mismatch.any():
    print(f"[WARN] day_night 컬럼과 계산값 불일치 행 수 = {mismatch.sum()} (계산값 기준으로 집계합니다)")

# 사양: 20250926 야간 생산 제외 -> shift_date가 20250926인 night 포함분 제거
df = df[~df["shift_date"].isin(EXCLUDE_SHIFT_DATES)].copy()

# 대상 교대일자만 남김
df = df[df["shift_date"].isin(TARGET_SHIFT_DATES)].copy()

# 정렬(사양 우선순위: end_day -> end_time)
df = df.sort_values(["station", "shift_date", "day_night_calc", "end_day", "end_time"]).reset_index(drop=True)

print("rows after shift-date filter =", len(df))
df.head(20)


rows after shift-date filter = 6442


,id,station,end_day,end_time,result,goodorbad,end_ts,day_night_calc,shift_date,day_night_src
0,5950942,Vision1,20250927,08:32:48,PASS,GoodFile,2025-09-27 08:32:48,day,20250927,NaN
1,5950941,Vision1,20250927,08:37:17,PASS,GoodFile,2025-09-27 08:37:17,day,20250927,NaN
2,5950940,Vision1,20250927,08:37:33,PASS,GoodFile,2025-09-27 08:37:33,day,20250927,NaN
3,5950939,Vision1,20250927,08:37:54,PASS,GoodFile,2025-09-27 08:37:54,day,20250927,NaN
4,5950938,Vision1,20250927,08:38:08,PASS,GoodFile,2025-09-27 08:38:08,day,20250927,NaN
5,5950951,Vision1,20250927,08:38:30,PASS,GoodFile,2025-09-27 08:38:30,day,20250927,NaN
6,5950950,Vision1,20250927,08:38:44,PASS,GoodFile,2025-09-27 08:38:44,day,20250927,NaN
7,5950949,Vision1,20250927,08:39:11,PASS,GoodFile,2025-09-27 08:39:11,day,20250927,NaN
8,5950948,Vision1,20250927,08:39:25,PASS,GoodFile,2025-09-27 08:39:25,day,20250927,NaN
9,5950947,Vision1,20250927,08:39:46,PASS,GoodFile,2025-09-27 08:39:46,day,20250927,NaN


In [6]:
# ============================================
# 4) 생산량(amount) 계산 (station별 + (shift_date, day/night)별)
# ============================================

amount_df = (
    df.groupby(["station", "shift_date", "day_night_calc"], as_index=False)
      .agg(amount=("id", "count"))
)

amount_df.sort_values(["shift_date", "day_night_calc", "station"]).reset_index(drop=True)


,station,shift_date,day_night_calc,amount
0,Vision1,20250927,day,1432
1,Vision2,20250927,day,1678
2,Vision1,20250927,night,1648
3,Vision2,20250927,night,1678
4,Vision1,20250928,night,4
5,Vision2,20250928,night,2


In [7]:
# ============================================
# 5) CT(delta seconds) 계산 + Boxplot 통계 + 대표 CT 산출
#    - 현재행 - 이전행 (첫 행은 NaN)
#    - 600초 이상 제외
#    - IQR 기반 lower/upper outlier 계산
#    - 대표 CT = mean(q1, median, q3)
# ============================================

def boxplot_stats_from_series(x: pd.Series) -> dict:
    x = x.dropna().astype(float)
    x = x[(x > 0) & (x < CT_MAX_SEC_EXCLUSIVE)]
    if len(x) == 0:
        return {"n": 0, "lower_outlier": np.nan, "q1": np.nan, "median": np.nan, "q3": np.nan, "upper_outlier": np.nan, "CT": np.nan}
    q1 = float(np.quantile(x, 0.25, method="linear"))
    med = float(np.quantile(x, 0.50, method="linear"))
    q3 = float(np.quantile(x, 0.75, method="linear"))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    ct = float(np.mean([q1, med, q3]))
    return {
        "n": int(len(x)),
        "lower_outlier": float(lower),
        "q1": float(q1),
        "median": float(med),
        "q3": float(q3),
        "upper_outlier": float(upper),
        "CT": float(ct),
    }

# 그룹별 delta 계산
df["ct_sec"] = (
    df.groupby(["station", "shift_date", "day_night_calc"])["end_ts"]
      .diff()
      .dt.total_seconds()
      .round(2)
)

stats_rows = []
for (station, shift_date, day_night), g in df.groupby(["station", "shift_date", "day_night_calc"]):
    st = boxplot_stats_from_series(g["ct_sec"])
    stats_rows.append({
        "station": station,
        "end_day": shift_date,        # 출력 요구 컬럼명(end_day)에 shift_date(D-DAY) 사용
        "day_night": day_night,
        "amount": int(len(g)),        # 사양: 해당 교대에 해당하는 행 갯수
        "CT": (np.nan if pd.isna(st["CT"]) else round(st["CT"], 2)),
        # 참고용(검증/디버그 목적) - 필요 없으면 삭제 가능
        "ct_n": st["n"],
        "lower_outlier": (np.nan if pd.isna(st["lower_outlier"]) else round(st["lower_outlier"], 2)),
        "q1": (np.nan if pd.isna(st["q1"]) else round(st["q1"], 2)),
        "median": (np.nan if pd.isna(st["median"]) else round(st["median"], 2)),
        "q3": (np.nan if pd.isna(st["q3"]) else round(st["q3"], 2)),
        "upper_outlier": (np.nan if pd.isna(st["upper_outlier"]) else round(st["upper_outlier"], 2)),
    })

stats_df = pd.DataFrame(stats_rows)

# 모든 조합(2 station x 2 shift x N dates)을 빠짐없이 출력하고 싶으면 reindex
full_index = pd.MultiIndex.from_product(
    [STATIONS, TARGET_SHIFT_DATES, ["day", "night"]],
    names=["station", "end_day", "day_night"]
)
stats_df_full = (
    stats_df.set_index(["station", "end_day", "day_night"])
            .reindex(full_index)
            .reset_index()
)

# amount/CT가 NaN이면 0/NaN로 정리
stats_df_full["amount"] = stats_df_full["amount"].fillna(0).astype(int)

# id 부여(출력용)
stats_df_full.insert(0, "id", range(1, len(stats_df_full) + 1))

# 요구 출력 컬럼(필요 시 참고 컬럼도 유지)
out_cols = ["id", "station", "end_day", "day_night", "amount", "CT"]
summary_df = stats_df_full[out_cols].copy()

summary_df


,id,station,end_day,day_night,amount,CT
0,1,Vision1,20250927,day,1432,20.33
1,2,Vision1,20250927,night,1648,18.67
2,3,Vision1,20250928,day,0,NaN
3,4,Vision1,20250928,night,4,22.67
4,5,Vision2,20250927,day,1678,18.67
5,6,Vision2,20250927,night,1678,18.67
6,7,Vision2,20250928,day,0,NaN
7,8,Vision2,20250928,night,2,21.00


In [8]:
# ============================================
# 6) (옵션) CSV 저장
# ============================================
out_path = "0927_0928_ct_compare_summary.csv"
summary_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("saved:", out_path)


saved: 0927_0928_ct_compare_summary.csv
